# Portfolio Optimizer Personal Project

### Part 1: Data Extraction and Preparation

In [9]:
import requests # For HTTP requests to get data from web API
import numpy as np # For numerical computations and arrays
import json # For working with JSON data from API
import pandas as pd # For data manipulations and analysis
from datetime import datetime, timedelta # For working with dates and times
import sklearn # For machine learning

In [ ]:
# Creates datetime object and sets it to June 1st, 2024
end_date = datetime(2024, 6, 1)

# Calculate the start date by subtracting 5000 days from the end date
start_date = end_date - timedelta(days=5000)

# API key to authenticate with the Financial Modeling Prep API
API_KEY = "wUvM2M29ZVxHDvK8IRp2P7iyrT8uhQG4"

# Ticker of the stock
ticker = "AAPL"

# Build the API URL to request historical daily closing prices for the selected stock and date range
url_hist_stock_price = f"https://financialmodelingprep.com/stable/historical-price-eod/light?symbol={ticker}&from={start_date.date()}&to={end_date.date()}&apikey={API_KEY}"

# Request the data from Financial Modeling Prep API
response_hist_stock_price = requests.get(url_hist_stock_price)

# Parses the JSON response
json_response_hist_stock_price = response_hist_stock_price.json()

# Convert the JSON into a pandas DataFrame
df_stock_hist = pd.DataFrame(json_response_hist_stock_price)

# Show the first few rows of the DataFrame
print(df_stock_hist.head(10))

# Convert date to datetime and price/volume to float
df_stock_hist['date'] = pd.to_datetime(df_stock_hist['date'])
df_stock_hist['price'] = df_stock_hist['price'].astype(float)
df_stock_hist['volume'] = df_stock_hist['volume'].astype(float)

# Set the date columns as the index of the DataFrame (Useful for time-series operations)
df_stock_hist.set_index('date', inplace=True)

# Drop the symbol column
df_stock_hist.drop(columns=['symbol'], inplace=True)

# Assign closing price and volume columns to variables
closing_price = df_stock_hist[['price']]
i_volume = df_stock_hist[['volume']]

# Show the first few rows of the DataFrame
print(df_stock_hist.head(10))


  symbol        date   price    volume
0   AAPL  2024-05-31  192.25  75158300
1   AAPL  2024-05-30  191.29  49947941
2   AAPL  2024-05-29  190.29  53068016
3   AAPL  2024-05-28  189.99  52280100
4   AAPL  2024-05-24  189.98  36326975
5   AAPL  2024-05-23  186.88  51005924
6   AAPL  2024-05-22  190.90  34648547
7   AAPL  2024-05-21  192.35  42309401
8   AAPL  2024-05-20  191.04  44361300
9   AAPL  2024-05-17  189.87  41282925
             price      volume
date                          
2024-05-31  192.25  75158300.0
2024-05-30  191.29  49947941.0
2024-05-29  190.29  53068016.0
2024-05-28  189.99  52280100.0
2024-05-24  189.98  36326975.0
2024-05-23  186.88  51005924.0
2024-05-22  190.90  34648547.0
2024-05-21  192.35  42309401.0
2024-05-20  191.04  44361300.0
2024-05-17  189.87  41282925.0


### Part 2: Feature Engineering

In [ ]:
# Daily Return
daily_return = closing_price.pct_change()
# 5-Day Return
ret_5d = closing_price.pct_change(5)
# 10-Day Return
ret_10d = closing_price.pct_change(10)

# 5-Day Volatility (Std Dev of Returns)
vol_5d = closing_price.pct_change().rolling(window=5).std()

# 10-Day Volatility
vol_10d = (closing_price.pct_change().rolling(window=10).std())

# Momentum (10d)
momentum_10d = closing_price - closing_price.shift(10)

# SMA_10/SMA_50 Ratio
sma_10 = closing_price.rolling(window=10).mean()
sma_50 = closing_price.rolling(window=50).mean()
sma_ratio = sma_10/sma_50

# Z-score (20d)
rolling_mean = closing_price.rolling(window=20).mean()
rolling_std = closing_price.rolling(window=20).std()
z_score_20d = (closing_price - rolling_mean)/rolling_std

# RSI (14d)
delta = closing_price.diff()
gain = delta.where(delta > 0, 0.0)
loss = -delta.where(delta < 0, 0.0)

avg_gain = gain.rolling(window=14).mean()
avg_loss = loss.rolling(window=14).mean()

rs = avg_gain/avg_loss
rsi_14 = 100 - (100 / (1 + rs))

# Apply log base 10 to volume to reduce the scale and improve ML model performance
volume =i_volume.apply(np.log10)

pd.set_option('display.max_rows', None)